### FIXED TORCH PRUNING IN index_mapping.py in update_concat_index_mapping in constants.MAX_VALID_DIM

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import lora_transfer_pruning
from  lora_transfer_pruning.core.pruning_instrumentor import PruningInstrumentor
from transformers import AutoModelForCausalLM
import torch

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
MODEL = "meta-llama/Llama-3.1-8B-Instruct" 
DEVICE = "cuda:0"
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    #quantization_config=quantization_config,
    #dtype=torch.bfloat16,
    device_map=DEVICE,
    # cache_dir="/glazkov-dev/.cache",
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [3]:
from transformer_lens.model_bridge import TransformerBridge
import transformer_lens

bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model,
    dtype=torch.float16,
)

In [4]:
bridge

TransformerBridge(
  (embed): EmbeddingBridge(
    (hook_in): HookPoint(name='embed.hook_in')
    (hook_out): HookPoint(name='embed.hook_out')
    (_original_component): Embedding(128256, 4096)
  )
  (rotary_emb): RotaryEmbeddingBridge(
    (hook_in): HookPoint(name='rotary_emb.hook_in')
    (hook_out): HookPoint(name='rotary_emb.hook_out')
    (hook_cos): HookPoint(name='rotary_emb.hook_cos')
    (hook_sin): HookPoint(name='rotary_emb.hook_sin')
    (_original_component): LlamaRotaryEmbedding()
  )
  (blocks): ModuleList(
    (0): BlockBridge(
      (hook_in): HookPoint(name='blocks.0.hook_in')
      (hook_out): HookPoint(name='blocks.0.hook_out')
      (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
      (_original_component): LlamaDecoderLayer(
        (self_attn): PositionEmbeddingsAttentionBridge(
          (hook_in): HookPoint(name='blocks.0.attn.hook_in')
          (hook_out): HookPoint(name='blocks.0.attn.hook_out')
          (hook_attn_scores): HookPoint(name='blocks.0.

https://github.com/VainF/Torch-Pruning?tab=readme-ov-file#sparse-training-optional

In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL)
validation_dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="validation",
)
validation_dataset

Dataset({
    features: ['text'],
    num_rows: 3760
})

In [6]:
for i, text in enumerate(validation_dataset):
    print(f"{i}: {text}")
    if i > 10:
        break

0: {'text': ''}
1: {'text': ' = Homarus gammarus = \n'}
2: {'text': ''}
3: {'text': ' Homarus gammarus , known as the European lobster or common lobster , is a species of clawed lobster from the eastern Atlantic Ocean , Mediterranean Sea and parts of the Black Sea . It is closely related to the American lobster , H. americanus . It may grow to a length of 60 cm ( 24 in ) and a mass of 6 kilograms ( 13 lb ) , and bears a conspicuous pair of claws . In life , the lobsters are blue , only becoming " lobster red " on cooking . Mating occurs in the summer , producing eggs which are carried by the females for up to a year before hatching into planktonic larvae . Homarus gammarus is a highly esteemed food , and is widely caught using lobster pots , mostly around the British Isles . \n'}
4: {'text': ''}
5: {'text': ' = = Description = = \n'}
6: {'text': ''}
7: {'text': ' Homarus gammarus is a large crustacean , with a body length up to 60 centimetres ( 24 in ) and weighing up to 5 – 6 kilogram

In [7]:
CONTEXT_LENGTH = 256
NUM_EVAL_BLOCKS = 32 #(block=batch)
EVAL_BATCH_SIZE = 1

validation_text = "\n\n".join(
    text for text in validation_dataset["text"] if text.strip()
)
validation_tokens = tokenizer(
    validation_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0]

num_blocks = NUM_EVAL_BLOCKS
assert num_blocks > 0, "Validation split does not contain enough tokens"
evaluation_blocks = validation_tokens[: num_blocks * CONTEXT_LENGTH].reshape(
    num_blocks, CONTEXT_LENGTH
)
evaluation_blocks.shape

torch.Size([32, 256])

In [8]:
model.train()

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): EmbeddingBridge(
      (hook_in): HookPoint(name='embed.hook_in')
      (hook_out): HookPoint(name='embed.hook_out')
      (_original_component): Embedding(128256, 4096)
    )
    (layers): ModuleList(
      (0): BlockBridge(
        (hook_in): HookPoint(name='blocks.0.hook_in')
        (hook_out): HookPoint(name='blocks.0.hook_out')
        (hook_mlp_in): HookPoint(name='blocks.0.hook_mlp_in')
        (_original_component): LlamaDecoderLayer(
          (self_attn): PositionEmbeddingsAttentionBridge(
            (hook_in): HookPoint(name='blocks.0.attn.hook_in')
            (hook_out): HookPoint(name='blocks.0.attn.hook_out')
            (hook_attn_scores): HookPoint(name='blocks.0.attn.hook_attn_scores')
            (hook_pattern): HookPoint(name='blocks.0.attn.hook_pattern')
            (hook_hidden_states): HookPoint(name='blocks.0.attn.hook_hidden_states')
            (hook_result): HookPoint(name='blocks.0.attn.hook_resu

We don't want norms like LlamaRMSNorm with x/std(x)*weight being pruned.

In [9]:
ignored_params = []
for name, param in model.named_parameters():
    if "norm" in name:
        ignored_params.append(param)

In [10]:
import torch
import torch.nn as nn
import torch_pruning as tp

example_inputs = evaluation_blocks[:4].to(DEVICE)

def trace_forward(model, input_ids):
    return model(
        input_ids=input_ids,
        #use_cache=False,
        #return_dict=True,
    ).logits #for compatability with tp

DG = tp.DependencyGraph().build_dependency(
    model,
    example_inputs=example_inputs,
    forward_fn=trace_forward,
    ignored_params=ignored_params,
    unwrapped_parameters=list(zip(ignored_params, [0] * len(ignored_params)))
)


In [11]:
print("if no waring about unwrapped_params, that means the ignored_params are correctly set, and the pruning will not affect them.")

if no waring about unwrapped_params, that means the ignored_params are correctly set, and the pruning will not affect them.


~~unwrapped_parameters - parameters that is not in registered model.parameters()~~
unwrapped_parameters - parameters that just nn.Parameter like weight = nn.Parameter(torch.ones(5)) instead of nn.Linear()

!Torch pruning doesnt understand semantics of channels/rows of Parameter


Note: parameters setted via self.linear = nn.Linear() through `__setattr__`  
it should be nn.Parameter()

Can afford to find pruning groups only on primitive modules like nn.Linear that directly participate in computational graph, not LlamaMLP or composite layers.

But pruning in_channels doesn't have fanout effect

#hung ups

check dep graph manually

In [12]:
target = bridge.blocks[0].attn.q._original_component
root = DG.module2node[target]

visited = set()
stack = [root]
edges = 0

while stack:
    node = stack.pop()
    if node in visited:
        continue

    visited.add(node)

    for dep in node.dependencies:
        edges += 1
        if dep.target not in visited:
            stack.append(dep.target)

print("reachable nodes:", len(visited))
print("reachable edges:", edges)
print("total DG nodes:", len(DG.module2node))

reachable nodes: 3279
reachable edges: 7262
total DG nodes: 3279


In [13]:
target = bridge.blocks[0].mlp.down_proj._original_component

root = DG.module2node[target]

visited = set()
stack = [root]
edges = 0

while stack:
    node = stack.pop()
    if node in visited:
        continue

    visited.add(node)

    for dep in node.dependencies:
        edges += 1
        if dep.target not in visited:
            stack.append(dep.target)

print("reachable nodes:", len(visited))
print("reachable edges:", edges)
print("total DG nodes:", len(DG.module2node))

reachable nodes: 3279
reachable edges: 7262
total DG nodes: 3279


Test of changed faster has_pruning_op function, that work not O(n)

In [14]:
from torch_pruning.dependency.group import Group

original_has_pruning_op = Group.has_pruning_op

def fast_has_pruning_op(self, dep, idxs):
    if not hasattr(self, "_fast_seen_ops"):
        self._fast_seen_ops = {
            (
                id(old_dep.target),
                old_dep.handler,
                frozenset(old_idxs),
            )
            for old_dep, old_idxs in self._group
        }
        self._fast_checks = 0

    key = (
        id(dep.target),
        dep.handler,
        frozenset(idxs),
    )

    self._fast_checks += 1

    if self._fast_checks % 100_000 == 0:
        print(
            "checks:", self._fast_checks,
            "unique states:", len(self._fast_seen_ops),
            "raw group size:", len(self._group),
            flush=True,
        )

    if key in self._fast_seen_ops:
        return True

    # Сразу резервируем состояние: вызывающий код после False
    # добавит его в group и processing_stack.
    self._fast_seen_ops.add(key)
    return False

Group.has_pruning_op = fast_has_pruning_op

In [15]:
from torch_pruning.dependency.constants import MAX_VALID_DIM
print(MAX_VALID_DIM)

18446744073709551616


In [16]:
group = DG.get_pruning_group(bridge.blocks[0].attn.q._original_component, tp.prune_linear_in_channels, idxs=[2, 6, 9] )

In [17]:
group = DG.get_pruning_group(bridge.blocks[0].attn.q._original_component, tp.prune_linear_out_channels, idxs=[2, 6, 9] )

In [28]:
print(group) #all works fine now without endless cycles and so on??????


--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)), len(idxs)=3
[1] prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on _Reshape_1554(), len(idxs)=3
[2] prune_out_channels on _Reshape_1554() => prune_out_channels on _ElementWiseOp_1553(TransposeBackward0), len(idxs)=3
[3] prune_out_channels on _ElementWiseOp_1553(TransposeBackward0) => prune_out_channels on _Slice_1552(), len(idxs)=3
[4] prune_out_channels on _ElementWiseOp_1553(TransposeBackward0) => prune_ou

In [69]:
from torch_pruning.dependency.node import Node
q_proj_id = id(group[0][0].source) 
group[0]

(prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)), [2, 6, 9])

Debug and find problem edge in dep graph

In [29]:
from collections import Counter, defaultdict
from torch_pruning.dependency.group import Group

original_has_pruning_op = Group.has_pruning_op

tp_debug = {
    "seen": set(),
    "target_counts": Counter(),
    "handler_counts": Counter(),
    "max_idxs": {},
    "samples": defaultdict(list),
}

MAX_STATES = 100_000


def debug_has_pruning_op(self, dep, idxs):
    target_id = id(dep.target)
    handler_name = getattr(
        dep.handler,
        "__qualname__",
        repr(dep.handler),
    )

    canonical_idxs = frozenset(
        (x.idx, x.root_idx)
        if hasattr(x, "idx")
        else x
        for x in idxs
    )

    key = (
        target_id,
        handler_name,
        canonical_idxs,
    )

    if key in tp_debug["seen"]:
        return True

    tp_debug["seen"].add(key)
    tp_debug["target_counts"][target_id] += 1
    tp_debug["handler_counts"][handler_name] += 1

    old_max = tp_debug["max_idxs"].get(target_id, 0)
    if len(canonical_idxs) > old_max:
        tp_debug["max_idxs"][target_id] = len(canonical_idxs)

    if len(tp_debug["samples"][target_id]) < 5:
        tp_debug["samples"][target_id].append(
            list(canonical_idxs)[:20]
        )

    if len(tp_debug["seen"]) >= MAX_STATES:
        raise RuntimeError(
            f"Stopped after {MAX_STATES} unique dependency states"
        )

    return False


Group.has_pruning_op = debug_has_pruning_op

In [30]:
group = DG.get_pruning_group(bridge.blocks[0].attn.q._original_component, tp.prune_linear_out_channels, idxs=[2, 6, 9] )

In [31]:
tp_debug["target_counts"]

Counter({128624635538320: 1,
         128624635538272: 1,
         128624635538128: 1,
         128624635537456: 1,
         128624635537264: 1,
         128624635537120: 1,
         128624635536976: 1,
         128624635533904: 1,
         128624635533712: 1,
         128624635533568: 1,
         128624635533424: 1,
         128624635533280: 1,
         128624635533136: 1,
         128624635532992: 1,
         128624635532848: 1,
         128624635532704: 1,
         128624635529584: 1,
         128624635529392: 1,
         128624635529248: 1,
         128624635529104: 1,
         128624635528960: 1,
         128624635528816: 1,
         128624635528576: 1,
         128624635528384: 1,
         128624635528240: 1,
         128624635529680: 1,
         128624635529824: 1,
         128624635529968: 1,
         128624635530112: 1,
         128624635530400: 1,
         128624635530544: 1,
         128624635530688: 1,
         128624635530832: 1,
         128624635534000: 1,
         12862

In [32]:
try:
    group = DG.get_pruning_group(
        bridge.blocks[0].attn.q._original_component, 
        tp.prune_linear_out_channels, 
        idxs=[2, 6, 9] )
finally:
    Group.has_pruning_op = original_has_pruning_op

In [33]:
tp_debug['max_idxs'] #all indices the same, shape doesnt change

{128624635538320: 3,
 128624635538272: 3,
 128624635538128: 3,
 128624635537456: 3,
 128624635537264: 3,
 128624635537120: 3,
 128624635536976: 3,
 128624635533904: 3,
 128624635533712: 3,
 128624635533568: 3,
 128624635533424: 3,
 128624635533280: 3,
 128624635533136: 3,
 128624635532992: 3,
 128624635532848: 3,
 128624635532704: 3,
 128624635529584: 3,
 128624635529392: 3,
 128624635529248: 3,
 128624635529104: 3,
 128624635528960: 3,
 128624635528816: 3,
 128624635528576: 3,
 128624635528384: 3,
 128624635528240: 3,
 128624635529680: 3,
 128624635529824: 3,
 128624635529968: 3,
 128624635530112: 3,
 128624635530400: 3,
 128624635530544: 3,
 128624635530688: 3,
 128624635530832: 3,
 128624635534000: 3,
 128624635534144: 3,
 128624635534288: 3,
 128624635534432: 3,
 128624635534576: 3,
 128624635534720: 3,
 128624635534864: 3,
 128624635535008: 3,
 128624635535152: 3,
 128624635535440: 3,
 128624635535584: 3,
 128624635535872: 3,
 128624635536016: 3,
 128624635535344: 3,
 128624635536

In [47]:
node_by_id = {
    id(node): node
    for node in DG.module2node.values()
}

for target_id, count in tp_debug["target_counts"].most_common(30):
    node = node_by_id.get(target_id)

    print(
        "\ncount:", count,
        "max idxs:", tp_debug["max_idxs"].get(target_id),
        "\nnode:", node,
        "\nnode outputs:", node.outputs,
        "\nnode module:", node.module,
        "\nsamples:", tp_debug["samples"][target_id],
    )


count: 1 max idxs: 3 
node: <Node: (model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)))> 
node outputs: [<Node: (_Reshape_1554())>] 
node module: Linear(in_features=4096, out_features=4096, bias=False) 
samples: [[(6, 6), (9, 9), (2, 2)]]

count: 1 max idxs: 3 
node: <Node: (_Reshape_1554())> 
node outputs: [<Node: (_ElementWiseOp_1553(TransposeBackward0))>] 
node module: _Reshape_1554() 
samples: [[(6, 6), (9, 9), (2, 2)]]

count: 1 max idxs: 3 
node: <Node: (_ElementWiseOp_1553(TransposeBackward0))> 
node outputs: [<Node: (_Slice_1552())>, <Node: (_Slice_1558())>, <Node: (_ElementWiseOp_1548(MulBackward0))>] 
node module: _ElementWiseOp_1553(TransposeBackward0) 
samples: [[(6, 6), (9, 9), (2, 2)]]

count: 1 max idxs: 3 
node: <Node: (_ElementWiseOp_1548(MulBackward0))> 
node outputs: [<Node: (_ElementWiseOp_1547(AddBackward0))>] 
node module: _ElementWiseOp_1548(MulBackward0) 
samples:

rotary embedding operations:

In [70]:
node_by_id.get(q_proj_id).outputs[0].outputs[0].outputs[0].outputs

[<Node: (_ConcatOp_1550(None))>]

In [71]:
node_by_id.get(q_proj_id).outputs[0].outputs[0].outputs[1].outputs

[<Node: (_ElementWiseOp_1551(NegBackward0))>]

In [72]:
#concat_op
node_by_id.get(q_proj_id).outputs[0].outputs[0].outputs[1].outputs[0].outputs

[<Node: (_ConcatOp_1550(None))>]

In [73]:
concat_node = node_by_id.get(q_proj_id).outputs[0].outputs[0].outputs[1].outputs[0].outputs[0]

In [74]:
concat_node.module

_ConcatOp_1550(None)

In [75]:
concat_node.dependencies

[prune_out_channels on _ConcatOp_1550(None) => prune_out_channels on _ElementWiseOp_1551(NegBackward0),
 prune_out_channels on _ConcatOp_1550(None) => prune_out_channels on _Slice_1552(),
 prune_out_channels on _ConcatOp_1550(None) => prune_out_channels on _ElementWiseOp_1549(MulBackward0)]

In [76]:
concat_node.outputs[0]

<Node: (_ElementWiseOp_1549(MulBackward0))>

As we can see, every target (node B in edge A->B in dep graph) have visited only once. No endless cycles.

No strange indices, only [2, 6, 9].

In [77]:
from collections import deque
import torch.nn as nn

queue = deque([(concat_node, 0)])
visited = set()

while queue:
    node, depth = queue.popleft()

    if node in visited or depth > 30:
        continue
    visited.add(node)

    module = node.module
    name = DG._module2name.get(module)
    print(module)

    if isinstance(module, nn.Linear):
        print(
            "UPSTREAM LINEAR:",
            name,
            module,
            "depth:",
            depth,
        )
        continue

    for input_node in node.inputs:
        queue.append((input_node, depth + 1))

_ConcatOp_1550(None)
_ElementWiseOp_1551(NegBackward0)
_Slice_1552()
_Slice_1558()
_ElementWiseOp_1553(TransposeBackward0)
_Reshape_1554()
Linear(in_features=4096, out_features=4096, bias=False)
UPSTREAM LINEAR: model.layers.0._original_component.self_attn._original_component.q_proj._original_component Linear(in_features=4096, out_features=4096, bias=False) depth: 4


Check outputs and how we go from q to o_proj

In [79]:
from collections import deque
import torch.nn as nn

queue = deque([(concat_node, 0)])
visited = set()

while queue:
    node, depth = queue.popleft()

    if node in visited or depth > 30:
        continue
    visited.add(node)

    module = node.module
    name = DG._module2name.get(module)
    print(module)

    if isinstance(module, nn.Linear):
        print(
            "UPSTREAM LINEAR:",
            name,
            module,
            "depth:",
            depth,
        )
        continue

    for input_node in node.outputs:
        queue.append((input_node, depth + 1))



_ConcatOp_1550(None)
_ElementWiseOp_1549(MulBackward0)
_ElementWiseOp_1547(AddBackward0)
_ExpandOp_1546()
_ElementWiseOp_1545(CloneBackward0)
_Reshape_1524()
_ElementWiseOp_1523(BmmBackward0)
_Reshape_1522()
_ElementWiseOp_1521(MulBackward0)
_ElementWiseOp_1520(MaskedFillBackward0)
_ElementWiseOp_1519(ToCopyBackward0)
_ElementWiseOp_1518(SoftmaxBackward0)
_ElementWiseOp_1517(ToCopyBackward0)
_ExpandOp_1516()
_Reshape_1495()
_ElementWiseOp_1494(BmmBackward0)
_Reshape_1493()
_ElementWiseOp_1492(TransposeBackward0)
_ElementWiseOp_1491(CloneBackward0)
_Reshape_1490()
_Reshape_1488()
_ElementWiseOp_1487(MmBackward0)
Linear(in_features=4096, out_features=4096, bias=False)
UPSTREAM LINEAR: model.layers.0._original_component.self_attn._original_component.o_proj._original_component Linear(in_features=4096, out_features=4096, bias=False) depth: 22


Path to o proj is correct:


In [80]:
from collections import deque
import torch.nn as nn

def node_label(node):
    module = node.module
    name = DG._module2name.get(module)

    if name is not None:
        return name

    return repr(node)


queue = deque([(concat_node, 0)])
visited = {concat_node}

# child -> previous node на пути от concat_node
parent = {concat_node: None}

target = None

while queue:
    node, depth = queue.popleft()
    module = node.module
    name = DG._module2name.get(module)

    if (
        isinstance(module, nn.Linear)
        and name is not None
        and name.endswith("o_proj._original_component")
    ):
        target = node
        print("FOUND:", name, "depth:", depth)
        break

    if depth >= 30:
        continue

    for next_node in node.outputs:
        if next_node in visited:
            continue

        visited.add(next_node)
        parent[next_node] = node
        queue.append((next_node, depth + 1))

FOUND: model.layers.0._original_component.self_attn._original_component.o_proj._original_component depth: 22


In [81]:
if target is None:
    print("o_proj не найден")
else:
    path = []
    current = target

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()

    print("\nPATH concat → o_proj:\n")

    for i, node in enumerate(path):
        prefix = "└──" if i == len(path) - 1 else "├──"
        print(f"{prefix} [{i:02d}] {node_label(node)}")


PATH concat → o_proj:

├── [00] <Node: (_ConcatOp_1550(None))>
├── [01] <Node: (_ElementWiseOp_1549(MulBackward0))>
├── [02] <Node: (_ElementWiseOp_1547(AddBackward0))>
├── [03] <Node: (_ExpandOp_1546())>
├── [04] <Node: (_ElementWiseOp_1545(CloneBackward0))>
├── [05] <Node: (_Reshape_1524())>
├── [06] <Node: (_ElementWiseOp_1523(BmmBackward0))>
├── [07] <Node: (_Reshape_1522())>
├── [08] <Node: (_ElementWiseOp_1521(MulBackward0))>
├── [09] <Node: (_ElementWiseOp_1520(MaskedFillBackward0))>
├── [10] <Node: (_ElementWiseOp_1519(ToCopyBackward0))>
├── [11] <Node: (_ElementWiseOp_1518(SoftmaxBackward0))>
├── [12] <Node: (_ElementWiseOp_1517(ToCopyBackward0))>
├── [13] <Node: (_ExpandOp_1516())>
├── [14] <Node: (_Reshape_1495())>
├── [15] <Node: (_ElementWiseOp_1494(BmmBackward0))>
├── [16] <Node: (_Reshape_1493())>
├── [17] <Node: (_ElementWiseOp_1492(TransposeBackward0))>
├── [18] <Node: (_ElementWiseOp_1491(CloneBackward0))>
├── [19] <Node: (_Reshape_1490())>
├── [20] <Node: (_Reshape_

In [82]:
#torch.cat((-q₂, q₁))                ConcatOp
#     ↓
# rotate_half(q) * sin                Mul
#     ↓
# q*cos + rotate_half(q)*sin          Add
#     ↓
# Q @ Kᵀ                              Bmm #1
#     ↓
# scores * scaling                    Mul
#     ↓
# causal masking                      MaskedFill
#     ↓
# FP32 conversion                     ToCopy
#     ↓
# softmax                             Softmax
#     ↓
# original dtype                      ToCopy
#     ↓
# attention_weights @ V               Bmm #2
#     ↓
# transpose + contiguous + reshape    Transpose/Clone/Reshape
#     ↓
# o_proj                              Mm

Original component is q (NOW, before it was k)!

In [83]:
print(
    concat_node.grad_fn,
    getattr(concat_node.grad_fn, "_saved_dim", "missing"),
)

<CatBackward0 object at 0x74fbc1b5e7d0> 18446744073709551615


In [84]:
grad_fn = concat_node.grad_fn

print("type:", type(grad_fn))
print("name:", grad_fn.name())
print("next_functions:", grad_fn.next_functions)
print("metadata:", grad_fn.metadata)

type: <class 'CatBackward0'>
name: CatBackward0
next_functions: ((<NegBackward0 object at 0x74fbc1b5e8c0>, 0), (<SliceBackward0 object at 0x74fbc1b5e950>, 0))
metadata: {}


In [23]:
model.config

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_theta": 500000.0,
    "rope_type": "llama3"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.5.1",
  "use_cache": true,
  "vocab_size": 128256
}

In [ ]:
bridge.blocks[0].attn.q._original_component

Linear(in_features=4096, out_features=4096, bias=False)

In [29]:
bridge.blocks[0].attn.k._original_component

Linear(in_features=4096, out_features=1024, bias=False)

In [30]:
bridge.blocks[0].attn.v._original_component

Linear(in_features=4096, out_features=1024, bias=False)

That means, that concat operation - about k or v _ConcatOp_1536([0, 1024, 2048])

In [ ]:
model.config.num_attention_heads * model.config.head_dim #out size of q

4096

32 * 128 = 4096 for llama

In [ ]:
print(group)

In [ ]:
tp.pruner.BasePruner() #num_heads - mapping for every module q_proj, k_proj, v_proj that shows size of head in it's dim
#prune_head_dims
#prune_num_heads

In [86]:
print("prune q_proj group")
print(group)

prune q_proj group

--------------------------------
          Pruning Group
--------------------------------
[0] prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)), len(idxs)=3
[1] prune_out_channels on model.layers.0._original_component.self_attn._original_component.q_proj._original_component (Linear(in_features=4096, out_features=4096, bias=False)) => prune_out_channels on _Reshape_1554(), len(idxs)=3
[2] prune_out_channels on _Reshape_1554() => prune_out_channels on _ElementWiseOp_1553(TransposeBackward0), len(idxs)=3
[3] prune_out_channels on _ElementWiseOp_1553(TransposeBackward0) => prune_out_channels on _Slice_1552(), len(idxs)=3
[4] prune_out_channels on _ElementWiseOp_1553(TransposeBac

In [87]:
# 3. Do the pruning
if DG.check_pruning_group(group): # avoid over-pruning, i.e., channels=0.
    group.prune()
# 4. Save & Load
# model.zero_grad() # clear gradients to avoid a large file size
# torch.save(model, 'model.pth') # !! no .state_dict here since the structure has been changed after pruning
# model = torch.load('model.pth') # load the pruned model. you may need torch.load('model.pth', weights_only=False) for PyTorch 2.6.0+.


print pruning functions & layers

In [89]:
for i, (dep, idxs) in enumerate(group):
    layer = dep.layer
    pruning_fn = dep.pruning_fn
    print(layer, pruning_fn)

Linear(in_features=4096, out_features=4093, bias=False) <bound method LinearPruner.prune_out_channels of <torch_pruning.pruner.function.LinearPruner object at 0x74f76c1b0ee0>>
_Reshape_1554() <bound method DummyPruner.prune_out_channels of <torch_pruning.ops.ReshapePruner object at 0x74f76c1750c0>>
_ElementWiseOp_1553(TransposeBackward0) <bound method DummyPruner.prune_out_channels of <torch_pruning.ops.ElementWisePruner object at 0x74f76c1b1930>>
_Slice_1552() <bound method SlicePruner.prune_out_channels of <torch_pruning.ops.SlicePruner object at 0x74f76c175870>>
_Slice_1558() <bound method SlicePruner.prune_out_channels of <torch_pruning.ops.SlicePruner object at 0x74f76c175870>>
_ElementWiseOp_1548(MulBackward0) <bound method DummyPruner.prune_out_channels of <torch_pruning.ops.ElementWisePruner object at 0x74f76c1b1930>>
_ElementWiseOp_1547(AddBackward0) <bound method DummyPruner.prune_out_channels of <torch_pruning.ops.ElementWisePruner object at 0x74f76c1b1930>>
_ElementWiseOp_1

In [90]:
CONTEXT_LENGTH = 256
NUM_EVAL_BLOCKS = 32
EVAL_BATCH_SIZE = 1

validation_text = "\n\n".join(
    text for text in validation_dataset["text"] if text.strip()
)
validation_tokens = tokenizer(
    validation_text,
    add_special_tokens=False,
    return_tensors="pt",
).input_ids[0]

available_blocks = validation_tokens.numel() // CONTEXT_LENGTH
num_blocks = min(NUM_EVAL_BLOCKS, available_blocks)
assert num_blocks > 0, "Validation split does not contain enough tokens"
evaluation_blocks = validation_tokens[: num_blocks * CONTEXT_LENGTH].reshape(
    num_blocks, CONTEXT_LENGTH
)
evaluation_blocks.shape

torch.Size([32, 256])

In [91]:
from utils import evaluate_language_model
bridge.reset_hooks()
baseline_metrics = evaluate_language_model(
    bridge,
    evaluation_blocks,
    batch_size=EVAL_BATCH_SIZE,
)
baseline_metrics

RuntimeError: shape '[1, 256, -1, 128]' is invalid for input of size 1047808

RuntimeError: shape '[1, 256, -1, 128]' is invalid for input of size 1047808


Cant use torch pruning for 1 head. Should use for shapes that //head_dim

In [ ]:
#from docs in BasePruner!!!
#when prune_head_dims=True
#---------
#if ch_groups > 1:
    # if channel grouping is enabled, we repeat the pruning indices for each channel group.
                                # For example, w=[0,1,2,3,4,5,6,7,8] with groups=3, and the pruning indices are [0].
                                # We extend the indices as [0, 3, 6] to remove the first element in each group.
                                
#--------                        


#so we need to apply pruning by groups to remove some element in each group

~~TODO:~~ check does prune of Llama with q_proj will work (dont crush on .reshapes in attn) (NO!)